# A3 Diagnostic: GPT-4o-mini GSM8K Anomaly Investigation

**Anomaly:** GPT-4o-mini baseline accuracy dropped from 94% (indices 0-49) to 44% (indices 50-99), while all other models remained stable or improved.

**Protocol:**
1. Re-run GPT-4o-mini baseline on indices 50-99 (reproducibility check)
2. Re-run on subset of indices 0-49 (sanity check)
3. Analyze problem characteristics (complexity, answer magnitude)
4. If reproducible → genuine capability boundary; if not → transient API issue


In [ ]:
import json, os, time, random
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')

# ━━━ PATHS ━━━
BASE_DIR = Path("/content/drive/MyDrive/A3/E1aug_diag")
for d in ["data", "results"]:
    (BASE_DIR / d).mkdir(parents=True, exist_ok=True)

# ━━━ API SETUP ━━━
from openai import OpenAI
openai_client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

MODELS = {
    "gpt4omini": "gpt-4o-mini",
}

API_DELAY = 0.3
API_MAX_RETRIES = 5

print("✓ Setup complete")
print(f"  Output: {BASE_DIR}")


In [ ]:
from datasets import load_dataset
import re

def extract_gsm8k_answer(answer_text):
    match = re.search(r'####\s*([\d,]+(?:\.\d+)?)', answer_text)
    if match:
        return float(match.group(1).replace(',', ''))
    return None

# Load BOTH index ranges
ds = load_dataset("openai/gsm8k", "main", split="test")

problems_0_49 = []
for i in range(0, 50):
    item = ds[i]
    ans = extract_gsm8k_answer(item['answer'])
    problems_0_49.append({
        "index": i,
        "question": item['question'],
        "correct_answer": ans,
        "answer_text": item['answer'],
        "q_length": len(item['question'].split()),
    })

problems_50_99 = []
for i in range(50, 100):
    item = ds[i]
    ans = extract_gsm8k_answer(item['answer'])
    problems_50_99.append({
        "index": i,
        "question": item['question'],
        "correct_answer": ans,
        "answer_text": item['answer'],
        "q_length": len(item['question'].split()),
    })

# Problem complexity comparison
import numpy as np

def analyze_problems(problems, label):
    lengths = [p['q_length'] for p in problems]
    answers = [abs(p['correct_answer']) for p in problems if p['correct_answer'] is not None]
    steps = [p['answer_text'].count('\n') for p in problems]
    print(f"\n{label}:")
    print(f"  Question length: mean={np.mean(lengths):.0f}, median={np.median(lengths):.0f}, max={max(lengths)}")
    print(f"  Answer magnitude: mean={np.mean(answers):.0f}, median={np.median(answers):.0f}, max={max(answers):.0f}")
    print(f"  Solution steps: mean={np.mean(steps):.1f}, median={np.median(steps):.0f}, max={max(steps)}")

analyze_problems(problems_0_49, "Indices 0-49 (E1/E1b)")
analyze_problems(problems_50_99, "Indices 50-99 (E1aug)")

with open(BASE_DIR / "data" / "problems.json", "w") as f:
    json.dump({"idx_0_49": problems_0_49, "idx_50_99": problems_50_99}, f, indent=2)

print(f"\n✓ Loaded {len(problems_0_49)} + {len(problems_50_99)} problems")


In [ ]:
def call_gpt4omini(system, user, max_tokens=1024, temperature=0):
    for attempt in range(1, API_MAX_RETRIES + 1):
        try:
            resp = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                max_tokens=max_tokens,
                temperature=temperature,
            )
            time.sleep(API_DELAY)
            return resp.choices[0].message.content
        except Exception as e:
            wait = min(2 ** (attempt - 1), 30)
            print(f"  ⚠ attempt {attempt}: {e}. Retry in {wait}s")
            time.sleep(wait)
    return None

def parse_numerical(text):
    if text is None:
        return None
    # Try JSON first
    try:
        d = json.loads(text.strip())
        if isinstance(d, dict) and 'final' in d:
            val = d['final']
            if val is not None:
                return float(val)
    except:
        pass
    # Regex fallback
    import re
    m = re.search(r'(?:final[^\d]*?)([\d,]+\.?\d*)', text, re.IGNORECASE)
    if m and m.group(1).strip():
        try:
            return float(m.group(1).replace(',', ''))
        except ValueError:
            pass
    # Last number
    nums = re.findall(r'-?[\d,]+\.?\d+', text)
    if nums:
        try:
            return float(nums[-1].replace(',', ''))
        except ValueError:
            pass
    return None

print("✓ API helpers ready")


In [ ]:
from tqdm import tqdm

SYSTEM = "You are a math problem solver. Solve the problem and respond ONLY with JSON: {\"final\": <number>}"

results = {"run1_50_99": [], "run2_50_99": [], "sanity_0_49": []}

# ━━━ RUN 1: Re-run indices 50-99 ━━━
print("=== Run 1: GPT-4o-mini on indices 50-99 (reproducibility) ===")
for p in tqdm(problems_50_99, desc="Run 1 (50-99)"):
    user = f"Problem:\n{p['question']}\n\nSolve step by step, then give your final answer as JSON: {{\"final\": <number>}}"
    raw = call_gpt4omini(SYSTEM, user)
    answer = parse_numerical(raw)
    correct = (answer is not None and abs(answer - p['correct_answer']) < 0.01) if p['correct_answer'] is not None else False
    results["run1_50_99"].append({
        "index": p['index'],
        "correct_answer": p['correct_answer'],
        "model_answer": answer,
        "is_correct": correct,
        "raw": raw[:300] if raw else None,
    })

r1_correct = sum(1 for r in results["run1_50_99"] if r["is_correct"])
print(f"\nRun 1 result: {r1_correct}/50 = {r1_correct/50:.0%}")

# ━━━ RUN 2: Second run on 50-99 (variance check) ━━━
print("\n=== Run 2: GPT-4o-mini on indices 50-99 (temperature=0 variance) ===")
for p in tqdm(problems_50_99, desc="Run 2 (50-99)"):
    user = f"Problem:\n{p['question']}\n\nSolve step by step, then give your final answer as JSON: {{\"final\": <number>}}"
    raw = call_gpt4omini(SYSTEM, user)
    answer = parse_numerical(raw)
    correct = (answer is not None and abs(answer - p['correct_answer']) < 0.01) if p['correct_answer'] is not None else False
    results["run2_50_99"].append({
        "index": p['index'],
        "correct_answer": p['correct_answer'],
        "model_answer": answer,
        "is_correct": correct,
        "raw": raw[:300] if raw else None,
    })

r2_correct = sum(1 for r in results["run2_50_99"] if r["is_correct"])
print(f"\nRun 2 result: {r2_correct}/50 = {r2_correct/50:.0%}")

# ━━━ SANITY: Run on indices 0-49 ━━━
print("\n=== Sanity check: GPT-4o-mini on indices 0-49 ===")
for p in tqdm(problems_0_49, desc="Sanity (0-49)"):
    user = f"Problem:\n{p['question']}\n\nSolve step by step, then give your final answer as JSON: {{\"final\": <number>}}"
    raw = call_gpt4omini(SYSTEM, user)
    answer = parse_numerical(raw)
    correct = (answer is not None and abs(answer - p['correct_answer']) < 0.01) if p['correct_answer'] is not None else False
    results["sanity_0_49"].append({
        "index": p['index'],
        "correct_answer": p['correct_answer'],
        "model_answer": answer,
        "is_correct": correct,
        "raw": raw[:300] if raw else None,
    })

san_correct = sum(1 for r in results["sanity_0_49"] if r["is_correct"])
print(f"\nSanity result: {san_correct}/50 = {san_correct/50:.0%}")

with open(BASE_DIR / "results" / "diagnostic_trials.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\n{'='*60}")
print(f"SUMMARY:")
print(f"  Original E1aug (50-99): 22/50 = 44%")
print(f"  Re-run 1       (50-99): {r1_correct}/50 = {r1_correct/50:.0%}")
print(f"  Re-run 2       (50-99): {r2_correct}/50 = {r2_correct/50:.0%}")
print(f"  Sanity         (0-49):  {san_correct}/50 = {san_correct/50:.0%}")
print(f"  Original E1    (0-49):  47/50 = 94%")


In [ ]:
import numpy as np

# Per-problem consistency between runs
print("=== Per-problem consistency (Run 1 vs Run 2 on indices 50-99) ===")
agree = 0
both_right = 0
both_wrong = 0
r1_only = 0
r2_only = 0
for r1, r2 in zip(results["run1_50_99"], results["run2_50_99"]):
    if r1["is_correct"] == r2["is_correct"]:
        agree += 1
        if r1["is_correct"]:
            both_right += 1
        else:
            both_wrong += 1
    elif r1["is_correct"]:
        r1_only += 1
    else:
        r2_only += 1

print(f"  Agreement: {agree}/50 = {agree/50:.0%}")
print(f"    Both correct:  {both_right}")
print(f"    Both wrong:    {both_wrong}")
print(f"    Run1 only:     {r1_only}")
print(f"    Run2 only:     {r2_only}")

# Which problems are consistently hard?
print("\n=== Consistently wrong problems (both runs) ===")
hard_problems = []
for r1, r2 in zip(results["run1_50_99"], results["run2_50_99"]):
    if not r1["is_correct"] and not r2["is_correct"]:
        idx = r1["index"]
        p = next(p for p in problems_50_99 if p["index"] == idx)
        hard_problems.append(idx)
        print(f"  idx={idx}: answer={r1['correct_answer']}, q_words={p['q_length']}")

print(f"\n  → {len(hard_problems)} consistently hard problems")

# Prompt comparison: did E1aug use a different prompt?
print("\n=== DIAGNOSIS ===")
r1_acc = sum(1 for r in results['run1_50_99'] if r['is_correct']) / 50
r2_acc = sum(1 for r in results['run2_50_99'] if r['is_correct']) / 50
san_acc = sum(1 for r in results['sanity_0_49'] if r['is_correct']) / 50

if r1_acc < 0.6 and r2_acc < 0.6:
    print("  ✓ REPRODUCIBLE: 50-99 is genuinely harder for GPT-4o-mini")
    print("  → This is a capability boundary, not an API glitch")
    print("  → Recommendation: Report in paper as evidence of problem-set sensitivity")
elif r1_acc > 0.8 and r2_acc > 0.8:
    print("  ✗ NOT REPRODUCIBLE: Original run was likely a transient issue")
    print("  → Recommendation: Re-run E1aug consumer eval for gpt4omini/gsm8k")
else:
    print(f"  ? MIXED: Run1={r1_acc:.0%}, Run2={r2_acc:.0%}")
    print("  → High variance suggests model instability on these problems")
    print("  → Recommendation: Use median of 3 runs, or report with error bars")

if abs(san_acc - 0.94) > 0.15:
    print(f"\n  ⚠ Sanity check diverges from E1: {san_acc:.0%} vs 94%")
    print("  → Possible model version change or prompt sensitivity")
else:
    print(f"\n  ✓ Sanity check consistent with E1: {san_acc:.0%} ≈ 94%")

# Save summary
import numpy as np

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.bool_,)): return bool(obj)
        if isinstance(obj, (np.integer,)): return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

summary = {
    "original_e1_0_49": 0.94,
    "original_e1aug_50_99": 0.44,
    "rerun1_50_99": r1_acc,
    "rerun2_50_99": r2_acc,
    "sanity_0_49": san_acc,
    "agreement_rate": agree / 50,
    "consistently_hard_problems": hard_problems,
    "diagnosis": "reproducible" if (r1_acc < 0.6 and r2_acc < 0.6) else "not_reproducible" if (r1_acc > 0.8) else "mixed"
}

with open(BASE_DIR / "results" / "diagnostic_summary.json", "w") as f:
    json.dump(summary, f, indent=2, cls=NumpyEncoder)

print(f"\n✓ Saved to {BASE_DIR / 'results' / 'diagnostic_summary.json'}")
